# RGBD Range-Doppler Notebook

? `rgb.npy`?`depth.npy`?`mask.npy` ??????????
1. ?? Range-Doppler ?
2. ????? Range-Doppler ??? `.npy` ??

???RGB ?????????????????????? depth ? mask?


In [1]:
import sys, pathlib

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
from tqdm.auto import tqdm

repo_root = pathlib.Path.cwd().parents[0]
sys.path.insert(0, str(repo_root))

from witwin.radar import Radar
from witwin.radar.sigproc import process_rd
from examples.rgbd_range_doppler import RGBDSequence, build_interpolator


c:\Users\Asixa\miniconda3\envs\witwin2\Lib\site-packages\slangtorch\slangtorch.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
jitc_llvm_init(): LLVM API initialization failed ..


## 1. Input Paths


In [2]:
sequence_path = repo_root / 'output' / 'rfgen_rgbd_sequence.npz'
output_dir = repo_root / 'output' / 'rgbd_range_doppler_notebook'

if not sequence_path.exists():
    raise FileNotFoundError(
        f"{sequence_path} not found. Run: python -m examples.preprocess_rfgen_rd --input-dir E:\\Research2026\\RFGen1.7\\RFGen_RD"
    )

output_dir.mkdir(parents=True, exist_ok=True)


## 2. Load RGB / Depth / Mask


In [3]:
with np.load(sequence_path) as data:
    depths = np.asarray(data['depths'], dtype=np.float32)
    masks = np.asarray(data['masks'])
    rgb = np.asarray(data['rgb']) if 'rgb' in data.files else None
    source_fps = float(data['fps']) if 'fps' in data.files else 30.0
    frame_indices = np.asarray(data['frame_indices']) if 'frame_indices' in data.files else np.arange(depths.shape[0], dtype=np.int32)

if masks.dtype != np.bool_:
    if masks.ndim == 4 and masks.shape[-1] in {3, 4}:
        masks = np.any(masks[..., :3] > 0, axis=-1)
    else:
        masks = masks > 0

assert depths.ndim == 3, depths.shape
assert masks.ndim in {2, 3}, masks.shape
if masks.ndim == 2:
    masks = np.broadcast_to(masks[None, ...], depths.shape)
assert masks.shape == depths.shape, (masks.shape, depths.shape)
if rgb is not None:
    assert rgb.shape[0] == depths.shape[0], (rgb.shape, depths.shape)

sequence = RGBDSequence(depths=depths, pointclouds=None, masks=masks, fps=source_fps)
duration = depths.shape[0] / source_fps

print('sequence_path:', sequence_path)
print('rgb:', None if rgb is None else rgb.shape)
print('depths:', depths.shape, depths.dtype, float(depths.min()), float(depths.max()))
print('masks:', masks.shape, masks.dtype, int(masks.sum()))
print('frame_indices:', frame_indices.shape, int(frame_indices[0]), int(frame_indices[-1]))
print(f'duration: {duration:.2f}s @ {source_fps:.2f} fps')


sequence_path: e:\Code\witwin-platform\radar\output\rfgen_rgbd_sequence.npz
rgb: (1919, 128, 128, 3)
depths: (1919, 128, 128) float32 0.0 7.964750289916992
masks: (1919, 128, 128) bool 1687878
frame_indices: (1919,) 0 1918
duration: 63.97s @ 30.00 fps


## 3. Preview Input Frame


In [4]:
preview_idx = 0
num_cols = 3 if rgb is not None else 2
fig, axes = plt.subplots(1, num_cols, figsize=(5 * num_cols, 4))
if num_cols == 2:
    ax_depth, ax_mask = axes
else:
    ax_rgb, ax_depth, ax_mask = axes
    ax_rgb.imshow(rgb[preview_idx])
    ax_rgb.set_title('RGB Preview')
    ax_rgb.axis('off')

depth_im = ax_depth.imshow(depths[preview_idx], cmap='viridis')
ax_depth.set_title('Depth')
ax_depth.axis('off')
fig.colorbar(depth_im, ax=ax_depth, fraction=0.046, pad=0.04)

ax_mask.imshow(masks[preview_idx], cmap='gray')
ax_mask.set_title('Mask')
ax_mask.axis('off')
plt.tight_layout()


## 4. Radar Setup and Interpolator


In [5]:
radar_config = {
    "num_tx": 3,
    "num_rx": 4,
    "fc": 77e9,
    "slope": 60.012,
    "adc_samples": 256,
    "adc_start_time": 6,
    "sample_rate": 4400,
    "idle_time": 7,
    "ramp_end_time": 65,
    "chirp_per_frame": 128,
    "frame_per_second": 10,
    "num_doppler_bins": 128,
    "num_range_bins": 256,
    "num_angle_bins": 64,
    "power": 15,
    "tx_loc": [[0, 0, 0], [4, 0, 0], [2, 1, 0]],
    "rx_loc": [[-6, 0, 0], [-5, 0, 0], [-4, 0, 0], [-3, 0, 0]],
}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
backend = 'dirichlet' if device == 'cuda' else 'pytorch'
radar = Radar(radar_config, backend=backend, device=device)

rgbd_params = {
    "depth_scale": "auto",
    "pointcloud_scale": "auto",
    "pointcloud_convention": "camera",
    "mask_mode": "foreground-nonzero",
    "mask_buffer": 1,
    "depth_min": 0.10,
    "depth_max": 20.0,
    "zero_fill": True,
    "pixel_stride": 2,
    "max_points": 4096,
    "fx": None,
    "fy": None,
    "cx": None,
    "cy": None,
    "fov_deg": 70.0,
}

interpolator, total_time, source_frames, source_fps = build_interpolator(sequence, args=rgbd_params, device=radar.device)
print(f'backend={backend} device={radar.device}')
print(f'source_frames={source_frames}, source_fps={source_fps:.3f}, total_time={total_time:.3f}s')


c:\Users\Asixa\miniconda3\envs\witwin2\Lib\site-packages\setuptools\_distutils\_msvccompiler.py:12: UserWarning: _get_vc_env is private; find an alternative (pypa/distutils#340)
  warnings.warn(


backend=dirichlet device=cuda
source_frames=1919, source_fps=30.000, total_time=63.933s


## 5. Single-Frame RD


In [ ]:
chirp_period = (radar.config.idle_time + radar.config.ramp_end_time) * 1e-6
radar_valid_time = chirp_period * radar.config.num_tx * max(0, radar.config.chirp_per_frame - 1)
estimated_output_frames = max(0, int(np.floor((total_time - radar_valid_time) * radar.config.frame_per_second)) + 1)
if estimated_output_frames <= 0:
    raise ValueError(f"Sequence too short for one radar frame: total_time={total_time:.4f}s, radar_valid_time={radar_valid_time:.4f}s")

rd_num_frames = min(30, estimated_output_frames)
rd_start_frame = max(0, estimated_output_frames // 2 - rd_num_frames // 2)
selected_frame_indices = np.arange(rd_start_frame, rd_start_frame + rd_num_frames, dtype=int)

single_frame_idx = int(selected_frame_indices[0])
t0 = single_frame_idx / radar.config.frame_per_second
frame = radar.mimo(interpolator, t0=t0)
rd_db, _, ranges, velocities = process_rd(radar, frame, tx=0, rx=0, static_clutter_removal=True)
rd_db = rd_db[:, :len(ranges)]

print(f'preview radar frame: {single_frame_idx}')
print(f'continuous radar frames: {selected_frame_indices[0]} ... {selected_frame_indices[-1]}')

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(
    rd_db,
    extent=[float(ranges[0]), float(ranges[-1]), float(velocities[0]), float(velocities[-1])],
    origin="lower",
    aspect="auto",
    cmap="jet",
)
ax.set_title(f'Single-Frame RD (radar frame {single_frame_idx})')
ax.set_xlabel('Range (m)')
ax.set_ylabel('Velocity (m/s)')
fig.colorbar(im, ax=ax, label="Magnitude (dB)")
plt.tight_layout()
plt.show()


## 6. Full RD Sequence


In [ ]:
chirp_period = (radar.config.idle_time + radar.config.ramp_end_time) * 1e-6
radar_valid_time = chirp_period * radar.config.num_tx * max(0, radar.config.chirp_per_frame - 1)
total_time = sequence.depths.shape[0] / sequence.fps
max_output_frames = max(0, int(np.floor((total_time - radar_valid_time) * radar.config.frame_per_second)) + 1)
if max_output_frames <= 0:
    raise ValueError(f"Sequence too short for one radar frame: total_time={total_time:.4f}s, radar_valid_time={radar_valid_time:.4f}s")

rd_num_frames = min(30, max_output_frames)
rd_start_frame = max(0, max_output_frames // 2 - rd_num_frames // 2)
selected_frame_indices = np.arange(rd_start_frame, rd_start_frame + rd_num_frames, dtype=int)

print(f'source frames: {sequence.depths.shape[0]} @ {sequence.fps:.2f} fps')
print(f'radar fps: {radar.config.frame_per_second:.2f}')
print(f'total available rd frames: {max_output_frames}')
print(f'continuous radar frames: {selected_frame_indices[0]} ... {selected_frame_indices[-1]}')
print(f'frames selected for rendering: {len(selected_frame_indices)}')

rd_maps = []
for frame_idx in tqdm(selected_frame_indices, desc='Generating RD frames'):
    t0 = int(frame_idx) / radar.config.frame_per_second
    frame = radar.mimo(interpolator, t0=t0)
    rd_db, _, ranges, velocities = process_rd(radar, frame, tx=0, rx=0, static_clutter_removal=True)
    rd_maps.append(rd_db[:, :len(ranges)].astype(np.float32, copy=False))

rd_stack = np.stack(rd_maps, axis=0)
np.save(output_dir / 'rd_maps_db.npy', rd_stack)
np.save(output_dir / 'rd_frame_indices.npy', selected_frame_indices)
np.savez(output_dir / 'rd_axes.npz', ranges=np.asarray(ranges), velocities=np.asarray(velocities))
print('rd_stack:', rd_stack.shape)
print('saved:', output_dir / 'rd_maps_db.npy')


source frames: 1919 @ 30.00 fps
radar fps: 10.00
rd frames to generate: 640


## 7. RD Animation and Save GIF


In [ ]:
vmin = float(np.percentile(rd_stack, 5))
vmax = float(np.percentile(rd_stack, 99))
fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(
    rd_stack[0],
    extent=[float(ranges[0]), float(ranges[-1]), float(velocities[0]), float(velocities[-1])],
    origin="lower",
    aspect="auto",
    cmap="jet",
    vmin=vmin,
    vmax=vmax,
)
ax.set_xlabel('Range (m)')
ax.set_ylabel('Velocity (m/s)')
cbar = fig.colorbar(im, ax=ax, label="Magnitude (dB)")

def update(frame_pos):
    im.set_data(rd_stack[frame_pos])
    src_idx = int(selected_frame_indices[frame_pos])
    ax.set_title(f"RD Frame {frame_pos + 1}/{rd_stack.shape[0]} (radar frame {src_idx})")
    return [im]

ani = animation.FuncAnimation(fig, update, frames=rd_stack.shape[0], interval=1000 / max(1.0, radar.config.frame_per_second), blit=False)
gif_path = output_dir / 'rd_video.gif'
ani.save(gif_path, writer=animation.PillowWriter(fps=max(1, min(rd_stack.shape[0], int(round(radar.config.frame_per_second))))))
plt.close(fig)
gif_path


WindowsPath('e:/Code/witwin-platform/radar/output/rgbd_range_doppler_notebook/rd_video.gif')

In [ ]:
HTML(ani.to_jshtml())
